# Iterations of SuSiE-RSS and SuSiE-rRSS (data processing)

`niter` of SuSiE-RSS and SuSiE-rRSS in three simulations, from `10` `[niter_summary]` (`2026_output/niter_summary/<setting>_niter.rds`, one row per fit):

| setting | SuSiE-RSS | SuSiE-rRSS | panel | NCP | `max_iter` |
|---|---|---|---|---|---|
| Allele flip (Fig 1B / S2; `02` `Fine_Flip`, `25` `Fine_Flip_rRSS`) | `flipped_res` | `flipped_rrss` | flip rate | 800 × h2g | 50 |
| GTEx population mismatch (Fig 3A; `09` `Gtex_fine`) | `<ref>` | `<ref>_mix` | LD reference | 800 × h2g | 50 |
| UKB sketch (Fig 2B; `05` `UKBB_B_finemap`) | `fit_sketch` | `fit_sketch_mix` | sketch dimension B | 300,000 × h2g | 100 (susieR default) |

`ser_region` marks regions where SuSiE-rRSS falls back to the SER model (`R_reliability_flag`); the figure leaves them out for both methods. SuSiE-rRSS stops on PIP convergence, SuSiE-RSS on the ELBO.

**Outputs:**
- `figure_data/Supp.Niter_SuSiE_RSS_vs_rRSS.rds` (+ csv): one row per fit
- `figure_data/Supp.Niter_SuSiE_RSS_vs_rRSS_Summary.csv`: per setting, panel and method

In [1]:
source("../../R/utils.R")

data_name <- "Supp.Niter_SuSiE_RSS_vs_rRSS"
niter_dir <- "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/niter_summary"    # 10 [niter_summary]

## result folder -> setting, GWAS sample size (NCP = n x h2g) and max_iter of the fits
settings <- tribble(
  ~source,          ~setting,                    ~n_gwas, ~max_iter,
  "Fine_Flip",      "Allele flip",               800,     50,
  "Fine_Flip_rRSS", "Allele flip",               800,     50,
  "Gtex_fine",      "GTEx population mismatch",  800,     50,
  "UKBB_B_finemap", "UKB sketch",                300000,  100)

## the two fits compared per setting
ref_lab   <- c(insample = "In-Sample", ADSP = "ADSP", UKB = "UKB", kg = "1000 Genomes")
fits_rss  <- c("flipped_res",  names(ref_lab),                "fit_sketch")
fits_rrss <- c("flipped_rrss", paste0(names(ref_lab), "_mix"), "fit_sketch_mix")

In [2]:
niter_all <- map_dfr(list.files(niter_dir, pattern = "_niter\\.rds$", full.names = TRUE), readRDS) %>%
  rename(source = setting)
## every fit name per result folder (the ones used are fits_rss / fits_rrss)
niter_all %>% count(source, fit, L, has_R_diag) %>% as.data.frame()

niter <- niter_all %>%
  filter(L > 1, fit %in% c(fits_rss, fits_rrss)) %>%
  inner_join(settings, by = "source") %>%
  mutate(method  = if_else(fit %in% fits_rrss, "SuSiE-rRSS", "SuSiE-RSS"),
         ref     = str_remove(str_remove(fit, "^fit_"), "_(mix|rrss|res)$"),
         h2g     = as.numeric(str_match(file, "h2g_([0-9.]+(?:e-?[0-9]+)?)")[, 2]),
         ncp     = round(n_gwas * h2g, 1),
         ncausal = as.integer(str_match(file, "causal_(\\d+)")[, 2]),
         flip    = as.numeric(str_match(file, "miss_prop_([0-9.]+)")[, 2]),
         B_stoc  = as.integer(str_match(file, "B_stoc_(\\d+)")[, 2]),
         panel   = case_when(setting == "Allele flip" ~ paste0("Flip rate ", flip * 100, "%"),
                             setting == "UKB sketch"  ~ paste0("Sketch dim = ", B_stoc),
                             TRUE                     ~ unname(ref_lab[ref])),
         region  = paste(ref, str_remove(str_remove(file, "^.*/result/"), "_rrss\\.rds$|\\.rds$"), sep = ":"),  # same key, both methods
         flagged = method == "SuSiE-rRSS" & R_reliability_flag %in% TRUE,
         hit_max = niter >= max_iter)

## regions where SuSiE-rRSS falls back to the SER model (left out for both methods in the figure)
niter <- niter %>% group_by(setting, region) %>% mutate(ser_region = any(flagged)) %>% ungroup()

panel_lv <- c(paste0("Flip rate ", c(0.1, 1, 10), "%"), unname(ref_lab), paste0("Sketch dim = ", sort(unique(niter$B_stoc))))
niter <- niter %>% mutate(setting = factor(setting, levels = unique(settings$setting)),
                          panel   = factor(panel, levels = panel_lv))
stopifnot(!any(is.na(niter$panel)), !any(is.na(niter$ncp)), !any(is.na(niter$ncausal)))

## largest niter per setting: should equal max_iter
niter %>% group_by(setting, max_iter) %>% summarise(max_niter = max(niter, na.rm = TRUE), .groups = "drop") %>% as.data.frame()

## per setting, panel and method
niter_tab <- niter %>%
  group_by(setting, panel, method) %>%
  summarise(fits = n(), SER_fallback_regions = sum(ser_region), median_all = median(niter, na.rm = TRUE),
            median_no_SER = median(niter[!ser_region], na.rm = TRUE),
            pct_at_max_no_SER = round(100 * mean(hit_max[!ser_region]), 1), .groups = "drop")
as.data.frame(niter_tab)

source,fit,L,has_R_diag,n
<chr>,<chr>,<int>,<lgl>,<int>
Fine_Flip,flipped_res,10,FALSE,13500
Fine_Flip,kriging_res,10,FALSE,13500
Fine_Flip,original_res,10,FALSE,13500
Fine_Flip_rRSS,flipped_rrss,10,TRUE,13500
Fine_Flip_rRSS,kriging_rrss,10,TRUE,13500
Fine_Flip_rRSS,original_rrss,10,TRUE,13500
Gtex_fine,ADSP,10,FALSE,7125
Gtex_fine,ADSP_SER,1,FALSE,7125
Gtex_fine,ADSP_mix,10,TRUE,7125


setting,max_iter,max_niter
<fct>,<dbl>,<int>
Allele flip,50,50
GTEx population mismatch,50,50
UKB sketch,100,50


setting,panel,method,fits,SER_fallback_regions,median_all,median_no_SER,pct_at_max_no_SER
<fct>,<fct>,<chr>,<int>,<int>,<dbl>,<dbl>,<dbl>
Allele flip,Flip rate 0.1%,SuSiE-RSS,4500,3286,50,4,26.1
Allele flip,Flip rate 0.1%,SuSiE-rRSS,4500,3286,40,5,16.1
Allele flip,Flip rate 1%,SuSiE-RSS,4500,4013,50,50,62.0
Allele flip,Flip rate 1%,SuSiE-rRSS,4500,4013,50,22,35.3
Allele flip,Flip rate 10%,SuSiE-RSS,4500,4449,50,50,100.0
Allele flip,Flip rate 10%,SuSiE-rRSS,4500,4449,50,50,64.7
GTEx population mismatch,In-Sample,SuSiE-RSS,7125,1,4,4,0.0
GTEx population mismatch,In-Sample,SuSiE-rRSS,7125,1,4,4,0.3
GTEx population mismatch,ADSP,SuSiE-RSS,7125,69,4,4,1.5


In [3]:
save_data(niter, data_name)
write_csv(niter_tab, file.path(data_dir, paste0(data_name, "_Summary.csv")))